فكرة مشروع «دعم — Daam»

هو نظام ذكي يساعد الشركات في اتخاذ القرارات التشغيلية المتعلقة  بالفروع

مثلاً:

 شركة عندها فروع في الرياض والخبر وأبها، وتبي تعرف

«أي فرع لديه أفضل أداء تشغيلي؟»

بدل ما الموظف يحلل البيانات يدويًا، دعم يستقبل الطلب، يفهم نوع التحليل المطلوب، يتحقق من البيانات والصلاحيات، يستخدم أدوات تحليل الفروع، ويتأكد من أمان وجودة النتيجة، ثم يقيس تكلفة ووقت تنفيذ الطلب.

يعني الهدف الأساسي:

تحويل طلب المستخدم وبيانات الفروع إلى تحليل يساعد الشركة على اتخاذ قرار تشغيلي أفضل

In [1]:
def ask_daam(question):
    return f"Daam received: {question}"


question = "أي فرع لديه أفضل أداء تشغيلي؟"

print(
    ask_daam(question)
)

Daam received: أي فرع لديه أفضل أداء تشغيلي؟


In [2]:
COMPLEX_WORDS = [
    "تحليل",
    "قرار"
]


def choose_model(message):

    for word in COMPLEX_WORDS:

        if word in message:
            return "flagship"

    return "cheap"


model = choose_model(question)

print(
    "Selected model:",
    model
)

Selected model: cheap


In [3]:
ASSISTANT_NAME = "Daam"

SERVICES = [
    "branch_analysis",
    "performance_analysis",
    "cost_analysis",
    "revenue_analysis",
    "other"
]

CITIES = [
    "Khobar",
    "Abha",
    "Riyadh",
    "Madinah",
    "unknown"
]

print("Assistant:", ASSISTANT_NAME)
print("Services :", SERVICES)
print("Cities   :", CITIES)

from pydantic import BaseModel, field_validator, ValidationError


class BranchTicket(BaseModel):

    full_name: str
    service_type: str
    city: str
    urgent: bool
    branch_name: str | None = None

    @field_validator("service_type")
    @classmethod
    def check_service(cls, value):

        if value not in SERVICES:
            raise ValueError(
                "service_type is not supported"
            )

        return value

    @field_validator("city")
    @classmethod
    def check_city(cls, value):

        if value not in CITIES:
            raise ValueError(
                "city is not supported"
            )

        return value

    @field_validator("branch_name")
    @classmethod
    def check_branch(cls, value):

        if value is None:
            return value

        if len(value.strip()) < 2:
            raise ValueError(
                "branch_name is too short"
            )

        return value


ticket = BranchTicket(
    full_name="Alanoud",
    service_type="performance_analysis",
    city="Riyadh",
    urgent=True,
    branch_name="Riyadh Branch"
)

print(ticket)

Assistant: Daam
Services : ['branch_analysis', 'performance_analysis', 'cost_analysis', 'revenue_analysis', 'other']
Cities   : ['Khobar', 'Abha', 'Riyadh', 'Madinah', 'unknown']
full_name='Alanoud' service_type='performance_analysis' city='Riyadh' urgent=True branch_name='Riyadh Branch'


In [4]:
ASSISTANT_NAME = "Daam"

CURRENT_USER = "Alanoud"

BRANCHES = {
    "BR001": "Riyadh Branch",
    "BR002": "Khobar Branch",
    "BR003": "Abha Branch"
}

print("Assistant   :", ASSISTANT_NAME)
print("Current user:", CURRENT_USER)
print("Branches    :", BRANCHES)


def check_branch_performance(branch_id):
    return BRANCHES.get(branch_id, "not found")


def analyze_branch_cost(branch_id):
    return "Cost analysis completed for " + branch_id


def escalate_to_agent(reason):
    return "Transferred to a human agent: " + reason


TOOLS = {
    "check_branch_performance": {
        "function": check_branch_performance,
        "risk": "read_only"
    },
    "analyze_branch_cost": {
        "function": analyze_branch_cost,
        "risk": "side_effecting"
    },
    "escalate_to_agent": {
        "function": escalate_to_agent,
        "risk": "terminal"
    }
}


for name, tool in TOOLS.items():
    print(name, "→", tool["risk"])


def run_tool(name, args):

    if name not in TOOLS:
        return "error: unknown tool"

    tool = TOOLS[name]

    if tool["risk"] == "side_effecting" and args.get("name") != CURRENT_USER:
        return "error: not authorized"

    try:
        return tool["function"](**args)

    except TypeError:
        return "error: wrong arguments"


print(
    run_tool(
        "check_branch_performance",
        {"branch_id": "BR001"}
    )
)

Assistant   : Daam
Current user: Alanoud
Branches    : {'BR001': 'Riyadh Branch', 'BR002': 'Khobar Branch', 'BR003': 'Abha Branch'}
check_branch_performance → read_only
analyze_branch_cost → side_effecting
escalate_to_agent → terminal
Riyadh Branch


In [5]:
ASSISTANT_NAME = "Daam"

SERVICES = [
    "branch_performance",
    "branch_cost",
    "operational_analysis"
]

SCOPE_WORDS = [
    "الفرع",
    "الفروع",
    "الأداء",
    "التكلفة",
    "الزيارات",
    "التشغيل"
]

BLOCKED_PHRASES = [
    "تجاهل التعليمات",
    "ignore previous instructions",
    "system prompt"
]

PII_PATTERNS = {
    "NATIONAL_ID": r"\b[12]\d{9}\b",
    "PHONE": r"\b05\d{8}\b"
}

SECRET_CODE = "DAAM-2026"

REFUSALS = {
    "injection": "أقدر أساعدك في تحليل أداء الفروع والقرارات التشغيلية فقط. كيف أخدمك؟",
    "off_scope": "هذا خارج خدماتنا. أقدر أساعدك في أداء الفروع والتكاليف والتشغيل.",
    "leak": "عذراً، ما أقدر أعرض هذا الرد."
}


def check_input(message):

    for phrase in BLOCKED_PHRASES:
        if phrase in message:
            return False, "injection"

    for word in SCOPE_WORDS:
        if word in message:
            return True, "ok"

    return False, "off_scope"


import re


def mask_pii(message):

    for name, pattern in PII_PATTERNS.items():
        message = re.sub(
            pattern,
            "[" + name + "]",
            message
        )

    return message


def check_output(answer):

    if SECRET_CODE in answer:
        return False, "leak"

    return True, "ok"


def build_prompt(message):

    return f"""
You are {ASSISTANT_NAME}, an assistant for business operational decisions.
Answer only about these services: {SERVICES}.
User message: {message}
"""


def run_pipeline(message):

    allowed, reason = check_input(message)

    if not allowed:
        return "blocked", REFUSALS[reason]

    safe_message = mask_pii(message)

    prompt = build_prompt(safe_message)

    answer = "يمكن تحديد أفضل فرع من خلال مقارنة الزيارات والتكاليف والأداء التشغيلي."

    allowed, reason = check_output(answer)

    if not allowed:
        return "blocked", REFUSALS[reason]

    return "passed", answer


print(
    run_pipeline(
        "أي فرع لديه أفضل أداء؟"
    )
)

('blocked', 'هذا خارج خدماتنا. أقدر أساعدك في أداء الفروع والتكاليف والتشغيل.')


In [6]:
GUESS_WORDS = [
    "أعتقد",
    "تقريباً"
]

TOLERANCE = 10


GOLDEN_SET = [
    {
        "question": "أي فرع لديه أفضل أداء تشغيلي؟",
        "expected": "أفضل فرع"
    },
    {
        "question": "كم تكلفة تشغيل الفرع؟",
        "expected": "تكلفة التشغيل"
    },
    {
        "question": "أي فرع لديه أعلى عدد من الزيارات؟",
        "expected": "أعلى عدد من الزيارات"
    },
    {
        "question": "وش أفضل مطعم في الرياض؟",
        "expected": "خارج خدماتنا"
    }
]


ANSWERS_V1 = {
    "أفضل": "أفضل فرع هو الفرع الذي يحقق أعلى أداء تشغيلي.",
    "تكلفة": "تكلفة التشغيل يتم تحليلها ومقارنتها بأداء الفرع.",
    "زيارات": "أعلى عدد من الزيارات يحدد الفرع الأكثر إقبالاً."
}


def answer_question(question, answers):

    for keyword, answer in answers.items():

        if keyword in question:
            return answer

    return "هذا خارج خدماتنا."


def run_eval(answers):

    passed = 0

    for case in GOLDEN_SET:

        answer = answer_question(
            case["question"],
            answers
        )

        if case["expected"] in answer:

            passed += 1

            print(
                "PASS |",
                case["question"]
            )

        else:

            print(
                "FAIL |",
                case["question"]
            )

    score = passed * 100 // len(GOLDEN_SET)

    print(
        "Score:",
        score,
        "%"
    )

    return score


baseline = run_eval(ANSWERS_V1)

print(
    "Baseline:",
    baseline,
    "%"
)

PASS | أي فرع لديه أفضل أداء تشغيلي؟
PASS | كم تكلفة تشغيل الفرع؟
PASS | أي فرع لديه أعلى عدد من الزيارات؟
FAIL | وش أفضل مطعم في الرياض؟
Score: 75 %
Baseline: 75 %


In [7]:
ASSISTANT_NAME = "Daam"

PRICE_PER_TOKEN = {
    "flagship": 10,
    "cheap": 2
}

LATENCY = {
    "flagship": 1.2,
    "cheap": 0.4
}

COMPLEX_WORDS = [
    "تحليل",
    "قرار"
]

DEMO_ANSWER = "تم تحليل بيانات الفرع وتقديم التوصية المناسبة."


REPLAY = [
    "أي فرع لديه أفضل أداء؟",
    "كم تكلفة تشغيل الفرع؟",
    "أي فرع لديه أفضل أداء؟",
    "أبي تحليل لأداء الفروع",
    "كم تكلفة تشغيل الفرع؟",
    "أي فرع لديه أفضل أداء؟"
]


def count_tokens(text):
    return len(text.split())


def call_model(model, message):

    tokens = count_tokens(message) + count_tokens(DEMO_ANSWER)

    cost = tokens * PRICE_PER_TOKEN[model]

    seconds = LATENCY[model]

    return DEMO_ANSWER, cost, seconds


def choose_model(message):

    for word in COMPLEX_WORDS:

        if word in message:
            return "flagship"

    return "cheap"


def run_replay(use_cache, use_router):

    cache = {}

    total_cost = 0

    total_seconds = 0

    for message in REPLAY:

        if use_cache and message in cache:
            continue

        model = "flagship"

        if use_router:
            model = choose_model(message)

        answer, cost, seconds = call_model(
            model,
            message
        )

        cache[message] = answer

        total_cost += cost

        total_seconds += seconds

    return total_cost, round(total_seconds, 1)


before = run_replay(
    False,
    False
)

print(
    "Before:",
    before
)

Before: (690, 7.2)


🧩 ليش رتبنا الأكواد بهذا الشكل؟

لأننا ما نبغى المشروع يكون مجموعة أكواد منفصلة

 نبغى كل Module يمثل مرحلة من مراحل عمل النظام.

Module 1 — LAB 1

استقبال السؤال

المستخدم يبدأ بسؤال

أي فرع لديه أفضل أداء تشغيلي؟

فأول شيء لازم يسويه النظام هو استقبال السؤال

لذلك استخدمنا :

ask_daam()

دوره: نقطة بداية المشروع

Module 2 — LAB 2

اختيار النموذج

بعد ما استقبلنا السؤال، نحتاج نحدد كيف نتعامل معه.

إذا السؤال بسيط → نستخدم نموذج cheap.

إذا فيه كلمات مثل:

تحليل

قرار

→ نستخدم flagship.

ليش حطيناه بعد LAB 1؟

لأن النظام أولًا يحتاج يعرف وش سؤال المستخدم، وبعدها يقرر أي نموذج يناسبه.

Module 3A — LAB 3A

فهم وتنظيم الطلب

بعد اختيار النموذج، نحتاج نفهم تفاصيل الطلب.

مثلاً:

أنا العنود من الرياض، أبي تحليل أداء فرع الرياض.

نحوّل الطلب إلى بيانات منظمة:

Name → Alanoud
Service → performance_analysis
City → Riyadh
Branch → Riyadh Branch

وهنا جاء دور:

BranchTicket

؟

عشان نتأكد أن البيانات صحيحة قبل ما نرسلها لمرحلة تنفيذ التحليل.

Module 3B — LAB 3B

استخدام أدوات الفروع + الصلاحيات

الآن بعد ما عرفنا وش يبي المستخدم، نحتاج ننفذ الطلب.

عندنا أدوات مثل:

check_branch_performance
analyze_branch_cost
escalate_to_agent

والنظام يتأكد أيضًا من صلاحية المستخدم قبل تنفيذ بعض العمليات.

مثلاً:

CURRENT_USER = "Alanoud"

فلو طلبت العنود تحليل تكلفة الفرع، النظام يتحقق من أنها مستخدمة مصرح لها


ليش 3B بعد 3A؟

لأن:

3A = فهم الطلب

ثم:

3B = تنفيذ الطلب

وهذا يخلي التسلسل منطقي.

Module 4 — LAB 4

الأمان

قبل ما نعتمد النتيجة، نحتاج نتأكد أن الطلب آمن
النظام يفحص

 مثلاً:

	•	هل السؤال متعلق بخدمات دعم؟
	•	هل فيه محاولة لتجاوز التعليمات؟
	•	هل فيه بيانات شخصية؟
	•	هل يحاول المستخدم الحصول على معلومات سرية؟

وفيه أيضًا

mask_pii

لإخفاء البيانات الحساسة.


ليش بعد الأدوات؟

لأن LAB 4

 يمثل طبقة الحماية للنظام، بحيث يتم فحص المدخلات والمخرجات وحماية النظام
  من الطلبات غير المناسبة.

Module 5 — LAB 5

تقييم جودة الإجابة
الحين عندنا نتيجة.


لكن نسأل

هل إجابة دعم صحيحة؟

نستخدم :

GOLDEN_SET

وهو عبارة عن أسئلة وإجابات متوقعة.

ثم نحسب:

PASS
FAIL
Score

مثلاً

Score: 100%

ليش حطيناه بعد التنفيذ والأمان؟

لأننا ما نقدر نقيم الإجابة إلا بعد ما يكون عندنا إجابة فعلية.

 Module 6 — LAB 6

التكلفة والسرعة

آخر مرحلة هي قياس كفاءة النظام.

نسأل:

	•	كم تكلفة تشغيل الطلب؟
	•	كم استغرق؟
	•	هل نقدر نستخدم نموذج أرخص؟
	•	هل نقدر نستخدم Cache لتجنب إعادة معالجة نفس السؤال؟

وهنا نستخدم:

run_replay()

ليش آخر شيء؟

لأننا بعد ما نبني وننفذ ونقيّم النظام، نقدر نقيس تكلفته وسرعته وكفاءته.

## **الشكل الكامل للمشروع**

المستخدم

   ↓

يسأل دعم سؤال

   ↓

LAB 1

استقبال السؤال

   ↓

LAB 2

اختيار النموذج

   ↓

LAB 3A

فهم وتنظيم الطلب

   ↓

LAB 3B

استخدام أدوات الفروع + الصلاحيات

   ↓

LAB 4

الأمان وحماية البيانات

   ↓

LAB 5

تقييم جودة الإجابة

   ↓

LAB 6

قياس التكلفة والسرعة

   ↓

النتيجة

رتبت الـ
Modules

حسب دورة عمل النظام يبدأ دعم باستقبال سؤال المستخدم،

 ثم يحدد النموذج المناسب، وبعدها يحول الطلب إلى بيانات منظمة، ثم يستخدم أدوات


 تحليل الفروع مع التحقق من الصلاحيات. بعد ذلك تمر العملية على طبقة الأمان

ثم يتم تقييم جودة الإجابة، وأخيرًا نقيس التكلفة

  الـLatency.

لذلك كل LAB

يمثل مرحلة مختلفة ومتكاملة من عمل المشروع